In [7]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage 

#加载env文件
load_dotenv()

#初始化模型
model = init_chat_model(
    model = "qwen3.8-flash",
    model_provider="openai",
    api_key = os.getenv("api_key"),
    base_url = os.getenv("base_url")
)

#创建agent

agent = create_agent(
    model = model
)

In [8]:

#准备多模态消息
#老方法
# message = {
#     "role":"user",
#      "content":[
#          {"type": "text", "text": "描述一下图片内容"},
#          {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"}
#      ]
# }

#新方法
message = HumanMessage(
    [
         {"type": "text", "text": "描述一下图片内容"},
         {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"}
     ]
)


stream = agent.stream(
    {"messages":[message]},
    stream_mode="messages"
)

for chunk,metadata in stream:
    print(chunk.content,end='',flush=True)

这是一张在海滩上拍摄的温馨照片，画面主体是一位年轻女子和一只狗正在亲密互动。

女子侧身坐在松软的沙地上，留着一头棕色长发，身穿黑白相间的格子衬衫和深色长裤，赤着脚，手腕上戴着一块白色手表。她脸上洋溢着灿烂的笑容，眼睛微微眯起，神情十分愉悦放松。她伸出一只手，掌心向上，而那只浅黄色（米色）的拉布拉多犬则端坐在她对面，抬起一只前爪搭在她的手掌上，仿佛在与她"握手"或击掌；她的另一只手似乎正捏着小零食，像是在奖励狗狗。狗身上佩戴着一条印有彩色爪印图案的胸背带，连着一根红色的牵引绳，绳子随意地散落在沙地上。

背景是平静的海面与一道轻轻翻卷的浪花，天空明亮而柔和。画面右侧洒入一片温暖的金色阳光，形成朦胧的光晕，表明这应是日出或日落时分的"黄金时刻"，整张照片笼罩在暖调的光线中。

总体而言，这张图片捕捉了人与宠物之间信任、快乐与温情的瞬间，氛围轻松、治愈而美好。

In [10]:
from ipywidgets import FileUpload
from IPython.display import display

upload = FileUpload(accept="*",multiple=True,disabled=False,description="上传图片",icon="upload")
display(upload)

FileUpload(value=(), accept='*', description='上传图片', multiple=True)

In [5]:
print(upload.value)

()


In [8]:
#读取图片，转换成base64格式
import base64
#获取第一个（也是唯一一个）上传的文件
uploaded_file = upload.value[0]
#获取它的内存视图
content_mv = uploaded_file["content"]
#转换内存视图 -->  字节
img_bytes = bytes(content_mv)
#base64编码
img_b64 = base64.b64encode(img_bytes).decode('utf-8')



IndexError: tuple index out of range

In [ ]:
#组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64":img_b64,
        "mime_type":"image/jpeg"
    },
    {"type": "text", "text": "给我讲讲图中的内容，以火箭队喵喵的口吻"}
])

for chunk,metadata in agent.stream({"messages":[multimodal_question]},stream_mode="messages"):
    print(chunk.content,end='',flush=True)